## Benchmarking VectoreRAG + GraphRAG Recommnedation System

In [1]:
from pathlib import Path
import json

# Notebook is in agents/graphRAG/benchmarks -> go up 3 levels to galaxy-agent-xp-II
project_root = Path().resolve().parent.parent.parent

# Paths to JSON files
tools_path = project_root / "utilities/tools_metadata_downloader/data/galaxy_instance_tools_2025-12-04_23-58-00.json"
workflows_path = project_root / "utilities/workflow_downloader/data/galaxy_iwc_workflows_20251205_162934.json"


# Load JSON files
with open(tools_path, "r") as f:
    tools = json.load(f)

with open(workflows_path, "r") as f:
    workflows = json.load(f)

print(f"Loaded {len(tools)} tools and {len(workflows)} workflows.")


Loaded 14923 tools and 20 workflows.


In [2]:
# ---- Robust project root detection ----
from pathlib import Path
import sys

cwd = Path.cwd()
PROJECT_ROOT = None

for parent in [cwd] + list(cwd.parents):
    if (parent / "agents").exists():
        PROJECT_ROOT = parent
        break

if PROJECT_ROOT is None:
    raise RuntimeError("Project root not found")

sys.path.insert(0, str(PROJECT_ROOT))

In [3]:
from agents.graphRAG.pipeline.tool_retrieval_pipeline import ToolRetrievalPipeline
from agents.graphRAG.pipeline.workflow_retrival_pipeline import WorkflowRetrievalPipeline
from agents.ingestion.Load.neo4j_client import Neo4jClient
from agents.scripts.query_embedding import QueryEmbeddingService

# ---- Config path ----
config_path = PROJECT_ROOT / "agents/graphRAG/config/graph_db_config.yml"
print("Config exists:", config_path.exists())

# ---- Init ----
neo_client = Neo4jClient(config_path=str(config_path))
tool_pipeline = ToolRetrievalPipeline(neo_client)
workflow_pipeline = WorkflowRetrievalPipeline(neo_client)
embedder = QueryEmbeddingService()

/home/henok/Desktop/projects/galaxy-agent-xp-II/venv10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:agents.ingestion.Load.neo4j_client:Connected to Neo4j at bolt://localhost:7687 as neo4j


Config exists: True


INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (w:Workflow) ON (w.workflow_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (s:Step) ON (s.step_uid)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (t:Tool) ON (t.tool_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (c:Category) ON (c.category_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (i:ToolInput) ON (i.input_uid)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (o:ToolOutput) ON (o.output_uid)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (k:Keyword) ON (k.name)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX IF NOT EXISTS FOR (i:WorkflowInput) ON (i.input_id)
INFO:agents.ingestion.Load.neo4j_client:Index executed: CREATE INDEX 

 Loaded embedding model: BAAI/bge-base-en-v1.5


INFO:sentence_transformers.SentenceTransformer:Use pytorch device: cpu
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


 Loaded embedding model: BAAI/bge-base-en-v1.5


INFO:sentence_transformers.SentenceTransformer:Use pytorch device: cpu


 Loaded embedding model: BAAI/bge-base-en-v1.5


## tool benchmarking

In [4]:
from sklearn.metrics.pairwise import cosine_similarity

def compute_tool_recall_with_embedding(
    tool_pipeline,
    queries,
    ground_truth,
    embedder,
    k=5,
    threshold=0.8
):
    """
    Compute recall@k for tools using embedding similarity.

    Args:
        tool_pipeline: ToolRetrievalPipeline
        queries: list[str]
        ground_truth: dict[str, list[str]]
        embedder: QueryEmbeddingService
        k: int, top-k retrieved tools to consider
        threshold: float, cosine similarity threshold to count as match

    Returns:
        recall_scores: list[int]
        avg_recall: float
    """
    recall_scores = []
    total_recall = 0

    for query in queries:
        results = tool_pipeline.retrieve_tools(query, top_k=k)

        # Extract tool names safely
        retrieved_items = []
        for ctx in results[:k]:
            tool = ctx.get("tool", {})
            name = tool.get("tool_name") or tool.get("name") or tool.get("tool_id")
            if name:
                retrieved_items.append(name)

        retrieved_items = list(set(retrieved_items))
        expected_items = ground_truth.get(query, [])

        if not expected_items or not retrieved_items:
            recall = 0
        else:
            # Compute embeddings
            expected_embeds = embedder.embed_query(expected_items)
            retrieved_embeds = embedder.embed_query(retrieved_items)

            sim_matrix = cosine_similarity(expected_embeds, retrieved_embeds)
            max_sims = sim_matrix.max(axis=1)

            recall = 1 if any(s >= threshold for s in max_sims) else 0

        recall_scores.append(recall)
        total_recall += recall

        print(f"\nQuery: {query}")
        print(f"Expected: {expected_items}")
        print(f"Top-{k} retrieved: {retrieved_items}")
        print(f"Recall@{k} : {recall}")

    avg_recall = total_recall / len(queries)
    print(f"\nAverage Recall@{k} : {avg_recall:.2f}")

    return recall_scores, avg_recall

In [6]:
with open("tool_test_queries.json", "r") as f:
    dataset = json.load(f)

queries = dataset["queries"]
ground_truth = dataset["ground_truth"]

recall_scores, avg_recall = compute_tool_recall_with_embedding(
    tool_pipeline=tool_pipeline,
    queries=queries,
    ground_truth=ground_truth,
    embedder=embedder,
    k=5,
    threshold=0.8
)


Query: I want to search a sequence database for a query sequence using jackhmmer
Expected: ['jackhmmer']
Top-5 retrieved: ['phmmer', 'jackhmmer']
Recall@5 : 1

Query: tool to retrieve genomic datasets for completed Microbial Genome Projects from NCBI
Expected: ['Get Microbial Data']
Top-5 retrieved: ['NCBI Datasets Genomes', 'Get Microbial Data']
Recall@5 : 1

Query: convert genome coordinates or annotation files between different assembly versions
Expected: ['CrossMap Wig']
Top-5 retrieved: ['CrossMap GFF', 'CrossMap VCF']
Recall@5 : 0

Query: how can I translate gene identifiers between different organisms
Expected: ['gProfiler Orth']
Top-5 retrieved: ['annotateMyIDs']
Recall@5 : 0

Query: tool to download a list of URLs via lftp and create a collection
Expected: ['downloads']
Top-5 retrieved: ['downloads', 'FTP Link for Bioimage Archive', 'Online data']
Recall@5 : 1

Query: I need to extract reads from SRA archives using sam-dump
Expected: ['Extract reads from SRA']
Top-5 retrieved

## workflow benchmarking

In [7]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def compute_workflow_recall_with_embedding(
    workflow_pipeline,
    queries,
    ground_truth,
    embedder,
    k=5,
    threshold=0.8
):
    """
    Compute recall@k using embedding similarity between retrieved and expected workflows.

    Args:
        workflow_pipeline: WorkflowRetrievalPipeline
        queries: list[str]
        ground_truth: dict[str, list[str]]
        embedder: QueryEmbeddingService
        k: int
        threshold: float, cosine similarity threshold to count a match

    Returns:
        recall_scores: list[int]
        avg_recall: float
    """
    recall_scores = []
    total_recall = 0

    for query in queries:
        results = workflow_pipeline.retrieve_workflows(query, top_k=k)

        retrieved_names = []
        for ctx in results[:k]:
            wf = ctx.get("workflow", {})
            name = wf.get("workflow_name") or wf.get("name") or wf.get("workflow_id")
            if name:
                retrieved_names.append(name)

        retrieved_names = list(set(retrieved_names))
        expected_names = ground_truth.get(query, [])

        # Get embeddings for both expected and retrieved
        if not expected_names or not retrieved_names:
            recall = 0
        else:
            expected_embeds = embedder.embed_query(expected_names)
            retrieved_embeds = embedder.embed_query(retrieved_names)
            sim_matrix = cosine_similarity(expected_embeds, retrieved_embeds)
            max_sims = sim_matrix.max(axis=1)
            recall = 1 if any(s >= threshold for s in max_sims) else 0

        recall_scores.append(recall)
        total_recall += recall

        print(f"\nQuery: {query}")
        print(f"Expected: {expected_names}")
        print(f"Top-{k} retrieved: {retrieved_names}")
        print(f"Recall@{k}: {recall}")

    avg_recall = total_recall / len(queries)
    print(f"\nAverage Recall@{k}: {avg_recall:.2f}")

    return recall_scores, avg_recall

In [8]:
with open("workflow_test_queries.json", "r") as f:
    dataset = json.load(f)

queries = dataset["queries"]
ground_truth = dataset["ground_truth"]


recall_scores, avg_recall = compute_workflow_recall_with_embedding(
    workflow_pipeline=workflow_pipeline,
    queries=queries,
    ground_truth=ground_truth,
    embedder=embedder,  
    k=5,
    threshold=0.8  
)


Query: I want a genome assembly workflow using HiFi reads with HiC phasing following VGP4 standards
Expected: ['Genome Assembly from Hifi reads with HiC phasing - VGP4']
Top-5 retrieved: ['assembly-hifi-only-vgp3', 'kmer-profiling-hifi-vgp1', 'assembly-hifi-hic-phasing-vgp4', 'assembly-hifi-trio-phasing-vgp5', 'scaffolding-hic-vgp8']
Recall@5: 1

Query: workflow for scaffolding a genome using Bionano optical map data
Expected: ['Scaffolding with Bionano']
Top-5 retrieved: ['hi-c-contact-map-for-assembly-manual-curation', 'kmer-profiling-hifi-vgp1', 'assembly-hifi-trio-phasing-vgp5', 'scaffolding-bionano-vgp7', 'scaffolding-hic-vgp8']
Recall@5: 1

Query: how to perform k-mer profiling for PacBio HiFi trio data for VGP2
Expected: ['kmer-profiling-hifi-trio-VGP2']
Top-5 retrieved: ['assembly-hifi-only-vgp3', 'kmer-profiling-hifi-vgp1', 'assembly-hifi-hic-phasing-vgp4', 'assembly-hifi-trio-phasing-vgp5', 'kmer-profiling-hifi-trio-vgp2']
Recall@5: 1

Query: detect antimicrobial resistance 